# Mixed radix FFT

## Radix-2 SDF Butterfly

In [2]:
import numpy as np
import fxpmath as fxp
from scipy.fft import fft, ifft

In [3]:
class radix2_PreAdder:
    """
    A class used to represent a hardware preadder of a radix-2 butterfly

    ...

    Attributes
    ----------
    input_a : float
        upper input
    input_b : float
        lower input
    output_add : float
        adder output
    output_sub : float
        subtractor output

    Methods
    -------
    calculate(self)
        Calculates the adder and the subtractor outputs 
    """
    def __init__(self):
        self.input_a = 0.0
        self.input_b = 0.0
        self.output_add = 0.0
        self.output_sub = 0.0


    def calculate(self):
        self.output_add = self.input_a + self.input_b
        self.output_sub = self.input_a - self.input_b
        

class radix2_Rotator:
    cnt = 0
    def __init__(self, stage_index, num_of_stages):
        self.input = 0.0
        self.output = 0.0
        self.stage_index = stage_index
        self.twiddleROM = (np.ones(2**(num_of_stages-stage_index))).astype(complex)
        self.half_len = 2**(num_of_stages-stage_index-1)
        N = 2**num_of_stages
        for i in range(self.half_len):
            k = i * 2**(stage_index)
            self.twiddleROM[i+self.half_len] = np.exp(-1j*2*np.pi*k/N)
        
        # print(f"STAGE {stage_index}, twiddle = {self.twiddleROM}")

        # print(self.twiddleROM)

    def rotate(self, fifo_full_flag):
        if (fifo_full_flag): # dozvola da brojac vrti i cita redom twiddle faktore iz memorije
            # print(f"STAGE {self.stage_index}, FIFO FULL")
            self.output = self.input * self.twiddleROM[self.cnt]
            # print(f'STAGE {self.stage_index}, curr twiddle = {self.twiddleROM[self.cnt]}')
            if (self.cnt == self.half_len*2-1):
                self.cnt = 0
            else:
                self.cnt += 1
        

class Fifo:
    full = 0
    cnt = 0

    def __init__(self, depth):
        self.depth = depth
        self.buffer = (np.zeros(depth)).astype(complex)
        # print(self.depth)

    def is_full(self):
        return self.full
    
    def get_output(self):
        return self.buffer[-1]
    
    def shift(self, input_sample):
        self.cnt += 1
        self.buffer = np.roll(self.buffer, 1)
        self.buffer[0] = input_sample
        if (self.cnt > self.depth):
            self.full = 1
        else:
            self.full = 0



In [4]:
class radix2_SDF_stage:
    input_sample = 0.0
    output_sample = 0.0
    op_cnt = 0 # operation counter (counting how many add/subb operations are done)
    def __init__(self, stage_index, num_of_stages):
        self.stage_index = stage_index
        self.num_of_stages = num_of_stages
        self.num_of_samples = 2**(num_of_stages-stage_index)
        self.fifo = Fifo(2**(num_of_stages-stage_index-1))
        self.pre_adder = radix2_PreAdder()
        self.rotator = radix2_Rotator(stage_index=stage_index, num_of_stages=num_of_stages)

    def isFifoFull(self):
        return self.fifo.is_full()
    
    def calculate(self):
        # print(f'STAGE {self.stage_index}, op_cnt = ', self.op_cnt)
        self.pre_adder.input_a = self.fifo.get_output()
        self.pre_adder.input_b = self.input_sample
        self.pre_adder.calculate()

        # print(f'STAGE {self.stage_index}, add_out = {self.pre_adder.output_add}')
        # print(f'STAGE {self.stage_index}, sub_out = {self.pre_adder.output_sub}')
        
        if (self.op_cnt//(self.num_of_samples/2)): ## other half of the input stream is comming
            self.output_sample = self.pre_adder.output_add
            self.fifo.shift(self.pre_adder.output_sub) 
        else:
            self.output_sample = self.fifo.get_output()
            self.fifo.shift(self.input_sample)

        self.rotator.input = self.output_sample
        self.rotator.rotate(self.isFifoFull())
        self.output_sample = self.rotator.output

        # print(f'stage_{self.stage_index} : {self.output_sample}')

        if (self.op_cnt == self.num_of_samples-1):
            self.op_cnt = 0
        else:
            self.op_cnt += 1

        # print(f'STAGE {self.stage_index}, output = {self.output_sample}')


In [5]:
stage0 = radix2_SDF_stage(stage_index=0, num_of_stages=3)
stage1 = radix2_SDF_stage(stage_index=1, num_of_stages=3)
stage2 = radix2_SDF_stage(stage_index=2, num_of_stages=3)


# input_vector = [1.0, 2.0, 3.0, 4.0, 0.0, 0.0, 0.0]
# input_vector = [4.0, 3.0, 2.0, 1.0, 0.0, 0.0, 0.0, 0.0]

# input_vector = [1.0, 1.0, 0.0]
input_vector = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
fft_manual = []
for i in range(len(input_vector)):
    stage0.input_sample = input_vector[i]
    stage0.calculate()
    stage1.input_sample = stage0.output_sample
    stage1.calculate()
    stage2.input_sample = stage1.output_sample
    stage2.calculate()
    print(stage2.output_sample)
    if i > 6:
        fft_manual.append(stage2.output_sample)


0.0
0j
0j
0j
0j
0j
0j
(36+0j)
(-4+0j)
(-4+4j)
(-3.9999999999999996-4j)
(-4+9.65685424949238j)
(-3.9999999999999996-1.6568542494923797j)
(-4+1.6568542494923797j)
(-3.9999999999999987-9.65685424949238j)


In [6]:
def bracewell_buneman(xarray, length, log2length):
    ''' 
    bracewell-buneman bit reversal function
    inputs: xarray is array; length is array length; log2length=log2(length).
    output: bit reversed array xarray. 
    '''
    muplus = int((log2length+1)/2)
    mvar = 1
    reverse = np.zeros(length, dtype = int)
    upper_range = muplus+1
    for _ in np.arange(1, upper_range):
        for kvar in np.arange(0, mvar):
            tvar = 2*reverse[kvar]
            reverse[kvar] = tvar
            reverse[kvar+mvar] = tvar+1
        mvar = mvar+mvar
    if (log2length & 0x01):
            mvar = mvar/2

    mvar = int(mvar)
    for qvar in np.arange(1, mvar):
        
        nprime = qvar-mvar
        rprimeprime = reverse[qvar]*mvar
        for pvar in np.arange(0, reverse[qvar]):
            nprime = nprime+mvar
            rprime = rprimeprime+reverse[pvar]
            temp = xarray[nprime]
            xarray[nprime] = xarray[rprime]
            xarray[rprime] = temp
    return xarray

In [7]:
# print(fft_manual)
fft_manual = np.array(bracewell_buneman(fft_manual, len(fft_manual), int(np.log2(len(fft_manual)))))
print(fft_manual)

[36.+0.j         -4.+9.65685425j -4.+4.j         -4.+1.65685425j
 -4.+0.j         -4.-1.65685425j -4.-4.j         -4.-9.65685425j]


In [8]:
# input_vector = [1.0, 1.0]
input_vector = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0]
# input_vector = [1.0, 2.0, 3.0, 4.0]
fft_numpy = np.fft.fft(input_vector)
print(fft_numpy)

[36.+0.j         -4.+9.65685425j -4.+4.j         -4.+1.65685425j
 -4.+0.j         -4.-1.65685425j -4.-4.j         -4.-9.65685425j]


In [9]:
print(np.allclose(fft_manual,fft_numpy))

True


## Radix-3 SDF Butterfly 

In [10]:
class radix3_PreAdder:
    """
    A class used to represent a hardware preadder of a radix-3 butterfly

    ...

    Attributes
    ----------
    input_0 : float
        first input of preadder
    input_1 : float
        second input of preadder
    input_2 : float
        third input of preadder
    output_0 : float
        first output
    output_1 : float
        second output
    output_2 : float
        thirs output

    Methods
    -------
    calculate(self)
        Calculates the preadder outputs 
    """
    def __init__(self):
        self.input_0 = 0.0
        self.input_1 = 0.0
        self.input_2 = 0.0
        self.output_0 = 0.0
        self.output_1 = 0.0
        self.output_2 = 0.0


    def calculate(self):
        tmp_0_0 = self.input_0
        tmp_1_0 = self.input_1 + self.input_2
        tmp_2_0 = self.input_1 - self.input_2
        ###### prvi nivo pajplajna ^
        tmp_0_1 = tmp_0_0 + tmp_1_0
        tmp_1_1 = tmp_0_0 - (1/2)*tmp_1_0
        tmp_2_1 = tmp_2_0 * (-1j*np.sqrt(3)/2)
        ###### drugi nivo pajplajna ^
        tmp_0_2 = tmp_0_1
        tmp_1_2 = tmp_1_1 + tmp_2_1
        tmp_2_2 = tmp_1_1 - tmp_2_1
        ###### treci nivo pajplajna ^ (ovo su izlazni registri vrv)
        self.output_0 = tmp_0_2
        self.output_1 = tmp_1_2
        self.output_2 = tmp_2_2

class radix3_Rotator:
    cnt = 0
    def __init__(self, stage_index, num_of_stages, size):
        self.input = 0.0
        self.output = 0.0
        self.stage_index = stage_index
        # self.twiddleROM = (np.ones(3**(num_of_stages-stage_index))).astype(complex)
        self.twiddleROM = (np.ones(size)).astype(complex)
        # self.two_thirds_len = 3**(num_of_stages-stage_index) - (3**(num_of_stages-stage_index)//3)
        self.two_thirds_len = size - size//3
        # print(self.two_thirds_len)
        # N = 3**num_of_stages
        N = size
        if (self.two_thirds_len > 2):
            for i in range(self.two_thirds_len):
                if (i < self.two_thirds_len//2):
                    k = i * 3**(stage_index)
                else:
                    k = 2*(i-self.two_thirds_len//2) * 3**(stage_index)
                print("k = ", k)
                self.twiddleROM[i+(len(self.twiddleROM) - self.two_thirds_len)] = np.exp(-1j*2*np.pi*k/N)
        
        print(f"STAGE {stage_index}, twiddle = {self.twiddleROM}")

        # print(self.twiddleROM)

    def rotate(self, fifo_full_flag):
        if (fifo_full_flag): # dozvola da brojac vrti i cita redom twiddle faktore iz memorije
            # print(f"STAGE {self.stage_index}, FIFO FULL")
            self.output = self.input * self.twiddleROM[self.cnt]
            # print(f'STAGE {self.stage_index}, curr twiddle = {self.twiddleROM[self.cnt]}')
            if (self.cnt == len(self.twiddleROM)-1):
                self.cnt = 0
            else:
                self.cnt += 1

In [11]:
class radix3_SDF_stage:
    input_sample = 0.0
    output_sample = 0.0
    op_cnt = 0 # operation counter (counting how many add/subb operations are done)
    def __init__(self, stage_index, num_of_stages, size):
        self.stage_index = stage_index
        self.num_of_stages = num_of_stages
        # self.num_of_samples = 3**(num_of_stages-stage_index)
        self.num_of_samples = size
        # self.fifo_0 = Fifo(3**(num_of_stages-stage_index-1))
        # self.fifo_1 = Fifo(3**(num_of_stages-stage_index-1))
        self.fifo_0 = Fifo(size//3)
        self.fifo_1 = Fifo(size//3)
        self.pre_adder = radix3_PreAdder()
        self.rotator = radix3_Rotator(stage_index=stage_index, num_of_stages=num_of_stages, size=size)

    def isFifoFull_0(self):
        return self.fifo_0.is_full()
    def isFifoFull_1(self):
        return self.fifo_1.is_full()
    
    def calculate(self):
        # print(f'STAGE {self.stage_index}, op_cnt = ', self.op_cnt)
        self.pre_adder.input_0 = self.fifo_0.get_output()
        self.pre_adder.input_1 = self.fifo_1.get_output()
        self.pre_adder.input_2 = self.input_sample
        self.pre_adder.calculate()

        # print(f'STAGE {self.stage_index}, pre_adder_out_0 = {self.pre_adder.output_0}')
        # print(f'STAGE {self.stage_index}, pre_adder_out_1 = {self.pre_adder.output_1}')
        # print(f'STAGE {self.stage_index}, pre_adder_out_2 = {self.pre_adder.output_2}')
        
        if (self.op_cnt < (self.num_of_samples//3)): ## other half of the input stream is comming
            self.output_sample = self.fifo_0.get_output()
            self.fifo_0.shift(self.input_sample)
        elif ((self.op_cnt >= (self.num_of_samples//3)) and (self.op_cnt < (self.num_of_samples*2/3))):
            self.output_sample = self.fifo_1.get_output()
            self.fifo_1.shift(self.input_sample)
        else:
            self.output_sample = self.pre_adder.output_0
            self.fifo_0.shift(self.pre_adder.output_1)
            self.fifo_1.shift(self.pre_adder.output_2)

        self.rotator.input = self.output_sample
        self.rotator.rotate(self.isFifoFull_1())
        self.output_sample = self.rotator.output

        # print(f'stage_{self.stage_index} : {self.output_sample}')

        if (self.op_cnt == self.num_of_samples-1):
            self.op_cnt = 0
        else:
            self.op_cnt += 1

        # print(f'STAGE {self.stage_index}, output = {self.output_sample}')

In [12]:
stage0_radix3 = radix3_SDF_stage(stage_index=0, num_of_stages=3, size=27)
stage1_radix3 = radix3_SDF_stage(stage_index=1, num_of_stages=3, size=9)
stage2_radix3 = radix3_SDF_stage(stage_index=2, num_of_stages=3, size=3)

input_vector = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
input_vector = np.random.random(27)
input_vector_padded = np.append(input_vector, np.zeros(26))
fft_radix3_manual = []
for i in range(len(input_vector_padded)):
    stage0_radix3.input_sample = input_vector_padded[i]
    stage0_radix3.calculate()
    stage1_radix3.input_sample = stage0_radix3.output_sample
    stage1_radix3.calculate()
    stage2_radix3.input_sample = stage1_radix3.output_sample
    stage2_radix3.calculate()
    print(stage2_radix3.output_sample)
    if i >= 26:
        fft_radix3_manual.append(stage2_radix3.output_sample)

fft_radix3_manual = np.array(fft_radix3_manual)

k =  0
k =  1
k =  2
k =  3
k =  4
k =  5
k =  6
k =  7
k =  8
k =  0
k =  2
k =  4
k =  6
k =  8
k =  10
k =  12
k =  14
k =  16
STAGE 0, twiddle = [ 1.        +0.j          1.        +0.j          1.        +0.j
  1.        +0.j          1.        +0.j          1.        +0.j
  1.        +0.j          1.        +0.j          1.        +0.j
  1.        +0.j          0.97304487-0.23061587j  0.89363264-0.44879918j
  0.76604444-0.64278761j  0.59715859-0.80212319j  0.39607977-0.91821611j
  0.17364818-0.98480775j -0.05814483-0.99830816j -0.28680323-0.95798951j
  1.        +0.j          0.89363264-0.44879918j  0.59715859-0.80212319j
  0.17364818-0.98480775j -0.28680323-0.95798951j -0.68624164-0.72737364j
 -0.93969262-0.34202014j -0.99323836+0.11609291j -0.83548781+0.54950898j]
k =  0
k =  3
k =  6
k =  0
k =  6
k =  12
STAGE 1, twiddle = [ 1. +0.j         1. +0.j         1. +0.j         1. +0.j
 -0.5-0.8660254j -0.5+0.8660254j  1. +0.j        -0.5+0.8660254j
 -0.5-0.8660254j]
STAGE 2, twidd

In [13]:
import numpy as np

def digit_reverse_array(arr, radix):
    """
    Perform digit-reversal on a NumPy array for a given radix.
    
    Parameters:
        arr (np.ndarray): Input array to be reordered.
        radix (int): The radix (base) for the FFT.
    
    Returns:
        np.ndarray: Reordered array based on digit-reversal indices.
    """
    n = arr.size
    if not np.log(n) / np.log(radix) % 1 == 0:
        raise ValueError("The size of the array must be a power of the radix.")
    
    num_digits = int(np.log(n) / np.log(radix))
    
    def digit_reverse(index, radix, num_digits):
        reversed_index = 0
        for _ in range(num_digits):
            reversed_index = reversed_index * radix + (index % radix)
            index //= radix
        return reversed_index
    
    reordered = np.empty_like(arr)
    for i in range(n):
        reversed_index = digit_reverse(i, radix, num_digits)
        reordered[reversed_index] = arr[i]
    
    return reordered

def reverse_digit_reverse_array(arr, radix):
    """
    Reverse digit-reversal on a NumPy array for a given radix.
    
    Parameters:
        arr (np.ndarray): Input array that was digit-reversed.
        radix (int): The radix (base) used for digit-reversal.
    
    Returns:
        np.ndarray: Array restored to its original order.
    """
    n = arr.size
    if not (np.log(n) / np.log(radix)).is_integer():
        raise ValueError("The size of the array must be a power of the radix.")
    
    num_digits = int(np.log(n) / np.log(radix))
    
    def digit_reverse(index, radix, num_digits):
        reversed_index = 0
        for _ in range(num_digits):
            reversed_index = reversed_index * radix + (index % radix)
            index //= radix
        return reversed_index
    
    reordered = np.empty_like(arr)
    for i in range(n):
        original_index = digit_reverse(i, radix, num_digits)
        reordered[original_index] = arr[i]
    
    return reordered


In [14]:
# print(fft_radix3_manual)
fft_radix3_manual = reverse_digit_reverse_array(fft_radix3_manual, radix=3)
for num in fft_radix3_manual:
    print(num)
# fft_radix3_numpy = np.fft.fft([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0])
fft_radix3_numpy = np.fft.fft(input_vector)
print("\n")
for num in fft_radix3_numpy:
    print(num)
# print(fft_radix3_numpy)

(14.526101700987148+0j)
(-2.3684889474925925-1.7247237080789952j)
(-2.2906842104865297-0.7098359152922112j)
(0.6106377317173165+0.2621908451326481j)
(-0.08397000338251637+0.258610276871769j)
(-1.5424410292174335+0.21044336662908056j)
(0.6106377317173162-0.2621908451326486j)
(-0.9745514521877576-1.4472579910111982j)
(-1.2731980003556636-2.3264231931649824j)
(-0.5759464099866616+0.18050495115376486j)
(1.0698596623402996+0.9430886345680023j)
(0.2594410686287076+0.8485515700975437j)
(1.451697854556973-0.5865623025271437j)
(-1.2902384672281353+0.9009126987635688j)
(-0.9097358794806147-0.8182994237631658j)
(0.4144283587233003-0.989374208774736j)
(-1.3497094726552237+1.3958987809620558j)
(-0.2977090592541428+1.4055668070782805j)
(-0.5759464099866616-0.18050495115376486j)
(-1.8817763453401861+0.12539849317797752j)
(-0.3737595869239067-0.8617500271605356j)
(0.4144283587232992+0.9893742087747356j)
(-0.8738417169032906-0.8635279330275657j)
(0.2041267211841058+0.31186101452631326j)
(1.451697854556

In [15]:
twiddles = np.ones(9).astype(complex)
for i in range(6):
    k = i * 3**(0)
    print(k)
    twiddles[i+(9 - 6)] = np.exp(-1j*2*np.pi*k/9)

print(twiddles)


0
1
2
3
4
5
[ 1.        +0.j          1.        +0.j          1.        +0.j
  1.        +0.j          0.76604444-0.64278761j  0.17364818-0.98480775j
 -0.5       -0.8660254j  -0.93969262-0.34202014j -0.93969262+0.34202014j]


## Radix-5 SDF Butterfly 

In [16]:
class radix5_PreAdder:
    def __init__(self):
        self.input_0 = 0.0
        self.input_1 = 0.0
        self.input_2 = 0.0
        self.input_3 = 0.0
        self.input_4 = 0.0
        self.output_0 = 0.0
        self.output_1 = 0.0
        self.output_2 = 0.0
        self.output_3 = 0.0
        self.output_4 = 0.0


    def calculate(self):
        tmp_0_0 = self.input_0
        tmp_1_0 = self.input_1 + self.input_4
        tmp_2_0 = self.input_2 + self.input_3
        tmp_3_0 = self.input_1 - self.input_4
        tmp_4_0 = self.input_2 - self.input_3
        ###### prvi nivo pajplajna ^
        tmp_0_1 = tmp_0_0
        tmp_1_1 = tmp_1_0 + tmp_2_0
        tmp_2_1 = tmp_1_0 - tmp_2_0
        tmp_3_1 = tmp_3_0
        tmp_4_1 = tmp_4_0
        tmp_5_1 = tmp_3_0 + tmp_4_0 # dodatna grana izmedju
        ###### drugi nivo pajplajna ^
        tmp_0_2 = tmp_0_1 + tmp_1_1
        tmp_1_2 = tmp_0_1 + tmp_1_1 * (-0.25)
        tmp_2_2 = tmp_2_1 * 0.559
        tmp_3_2 = tmp_3_1 * (-1j*0.363)
        tmp_4_2 = tmp_4_1 * 1j*1.539
        tmp_5_2 = tmp_5_1 * (-1j*0.588)
        ###### treci nivo pajplajna ^
        tmp_0_3 = tmp_0_2
        tmp_1_3 = tmp_1_2 + tmp_2_2
        tmp_2_3 = tmp_1_2 - tmp_2_2
        tmp_3_3 = tmp_3_2 + tmp_5_2
        tmp_4_3 = tmp_4_2 + tmp_5_2
        ###### cetvrti nivo pajplajna ^
        self.output_0 = tmp_0_3
        self.output_1 = tmp_1_3 + tmp_3_3
        self.output_2 = tmp_2_3 + tmp_4_3
        self.output_4 = tmp_1_3 - tmp_3_3
        self.output_3 = tmp_2_3 - tmp_4_3

class radix5_Rotator:
    cnt = 0
    def __init__(self, stage_index, num_of_stages, size):
        self.input = 0.0
        self.output = 0.0
        self.stage_index = stage_index
        # self.twiddleROM = (np.ones(5**(num_of_stages-stage_index))).astype(complex)
        self.twiddleROM = (np.ones(size)).astype(complex)
        # self.four_fifths_len = 5**(num_of_stages-stage_index) - (5**(num_of_stages-stage_index)//5)
        self.four_fifths_len = size - (size//5)
        N = size
        if (self.four_fifths_len > 4):
            for i in range(self.four_fifths_len):
                if (i < self.four_fifths_len/4):
                    k = i * 5**(stage_index)
                elif ((i >= self.four_fifths_len/4) and (i < self.four_fifths_len/2)):
                    # UPITNO
                    k = 2*(i-self.four_fifths_len//4) * 5**(stage_index)
                elif ((i >= self.four_fifths_len/2) and (i < 3*self.four_fifths_len/4)):
                    # UPITNO
                    k = 3*(i-2*self.four_fifths_len//4) * 5**(stage_index)
                else:
                    k = 4*(i-3*self.four_fifths_len//4) * 5**(stage_index)
                print("k = ", k)
                self.twiddleROM[i+(len(self.twiddleROM) - self.four_fifths_len)] = np.exp(-1j*2*np.pi*k/N)
        
        print(f"STAGE {stage_index}, twiddle = {self.twiddleROM}")

        # print(self.twiddleROM)

    def rotate(self, fifo_full_flag):
        if (fifo_full_flag): # dozvola da brojac vrti i cita redom twiddle faktore iz memorije
            # print(f"STAGE {self.stage_index}, FIFO FULL")
            self.output = self.input * self.twiddleROM[self.cnt]
            # print(f'STAGE {self.stage_index}, curr twiddle = {self.twiddleROM[self.cnt]}')
            if (self.cnt == len(self.twiddleROM)-1):
                self.cnt = 0
            else:
                self.cnt += 1

In [17]:
class radix5_SDF_stage:
    input_sample = 0.0
    output_sample = 0.0
    op_cnt = 0 # operation counter (counting how many add/subb operations are done)
    def __init__(self, stage_index, num_of_stages, size):
        self.stage_index = stage_index
        self.num_of_stages = num_of_stages
        # self.num_of_samples = 5**(num_of_stages-stage_index)
        self.num_of_samples = size
        # self.fifo_0 = Fifo(5**(num_of_stages-stage_index-1))
        # self.fifo_1 = Fifo(5**(num_of_stages-stage_index-1))
        # self.fifo_2 = Fifo(5**(num_of_stages-stage_index-1))
        # self.fifo_3 = Fifo(5**(num_of_stages-stage_index-1))
        self.fifo_0 = Fifo(size//5)
        self.fifo_1 = Fifo(size//5)
        self.fifo_2 = Fifo(size//5)
        self.fifo_3 = Fifo(size//5)
        self.pre_adder = radix5_PreAdder()
        self.rotator = radix5_Rotator(stage_index=stage_index, num_of_stages=num_of_stages, size=size)

    def isFifoFull_3(self):
        return self.fifo_3.is_full()
    
    def calculate(self):
        self.pre_adder.input_0 = self.fifo_0.get_output()
        self.pre_adder.input_1 = self.fifo_1.get_output()
        self.pre_adder.input_2 = self.fifo_2.get_output()
        self.pre_adder.input_3 = self.fifo_3.get_output()
        self.pre_adder.input_4 = self.input_sample
        self.pre_adder.calculate()
        
        if (self.op_cnt < (self.num_of_samples//5)): ## other half of the input stream is comming
            self.output_sample = self.fifo_0.get_output()
            self.fifo_0.shift(self.input_sample)
        elif ((self.op_cnt >= (self.num_of_samples//5)) and (self.op_cnt < (self.num_of_samples*2/5))):
            self.output_sample = self.fifo_1.get_output()
            self.fifo_1.shift(self.input_sample)
        elif ((self.op_cnt >= (2*self.num_of_samples//5)) and (self.op_cnt < (self.num_of_samples*3/5))):
            self.output_sample = self.fifo_2.get_output()
            self.fifo_2.shift(self.input_sample)
        elif ((self.op_cnt >= (3*self.num_of_samples//5)) and (self.op_cnt < (self.num_of_samples*4/5))):
            self.output_sample = self.fifo_3.get_output()
            self.fifo_3.shift(self.input_sample)
        else:
            self.output_sample = self.pre_adder.output_0
            self.fifo_0.shift(self.pre_adder.output_1)
            self.fifo_1.shift(self.pre_adder.output_2)
            self.fifo_2.shift(self.pre_adder.output_3)
            self.fifo_3.shift(self.pre_adder.output_4)

        self.rotator.input = self.output_sample
        self.rotator.rotate(self.isFifoFull_3())
        self.output_sample = self.rotator.output

        # print(f'stage_{self.stage_index} : {self.output_sample}')

        if (self.op_cnt == self.num_of_samples-1):
            self.op_cnt = 0
        else:
            self.op_cnt += 1

        # print(f'STAGE {self.stage_index}, output = {self.output_sample}')

In [18]:
stage0_radix5 = radix5_SDF_stage(stage_index=0, num_of_stages=2, size=25)
stage1_radix5 = radix5_SDF_stage(stage_index=1, num_of_stages=2, size=5)

input_vector = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
input_vector = np.arange(25)
# input_vector = np.ones(25)
input_vector_padded = np.append(input_vector, np.zeros(24))
fft_radix5_manual = []
for i in range(len(input_vector_padded)):
    stage0_radix5.input_sample = input_vector_padded[i]
    stage0_radix5.calculate()
    stage1_radix5.input_sample = stage0_radix5.output_sample
    stage1_radix5.calculate()
    print(stage1_radix5.output_sample)
    if i >= 24:
        fft_radix5_manual.append(stage1_radix5.output_sample)

fft_radix5_manual = np.array(fft_radix5_manual)

k =  0
k =  1
k =  2
k =  3
k =  4
k =  0
k =  2
k =  4
k =  6
k =  8
k =  0
k =  3
k =  6
k =  9
k =  12
k =  0
k =  4
k =  8
k =  12
k =  16
STAGE 0, twiddle = [ 1.        +0.j          1.        +0.j          1.        +0.j
  1.        +0.j          1.        +0.j          1.        +0.j
  0.96858316-0.24868989j  0.87630668-0.48175367j  0.72896863-0.68454711j
  0.53582679-0.84432793j  1.        +0.j          0.87630668-0.48175367j
  0.53582679-0.84432793j  0.06279052-0.99802673j -0.42577929-0.90482705j
  1.        +0.j          0.72896863-0.68454711j  0.06279052-0.99802673j
 -0.63742399-0.77051324j -0.9921147 -0.12533323j  1.        +0.j
  0.53582679-0.84432793j -0.42577929-0.90482705j -0.9921147 -0.12533323j
 -0.63742399+0.77051324j]
STAGE 1, twiddle = [1.+0.j 1.+0.j 1.+0.j 1.+0.j 1.+0.j]
0.0
0.0
0.0
0.0
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
(300+0j)
(-12.5+17.205j)
(-12.5+4.065j)
(-12.5-4.065j)
(-12.5-17.205j)
(-12.499489407304694+98.94861736848772j)
(-12.500

In [19]:
# print(fft_radix3_manual)

fft_radix5_manual_rev = reverse_digit_reverse_array(fft_radix5_manual, radix=5)

print("Manual fft radix-5")
i = 0
for num in fft_radix5_manual_rev:
    print(f'{i} | {num:.2f}')
    i+=1
# fft_radix3_numpy = np.fft.fft([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0])
fft_radix5_numpy = np.fft.fft(input_vector)
print("\n")

print("Numpy fft")
i=0
for num in fft_radix5_numpy:
    print(f'{i} | {num:.2f}')
    i+=1
# print(fft_radix3_numpy)

Manual fft radix-5
0 | 300.00+0.00j
1 | -12.50+98.95j
2 | -12.49+48.69j
3 | -12.51+31.57j
4 | -12.50+22.74j
5 | -12.50+17.20j
6 | -12.50+13.31j
7 | -12.50+10.35j
8 | -12.50+7.93j
9 | -12.50+5.88j
10 | -12.50+4.07j
11 | -12.50+2.39j
12 | -12.50+0.79j
13 | -12.50-0.78j
14 | -12.50-2.37j
15 | -12.50-4.07j
16 | -12.50-5.89j
17 | -12.50-7.93j
18 | -12.50-10.35j
19 | -12.50-13.32j
20 | -12.50-17.20j
21 | -12.50-22.74j
22 | -12.51-31.57j
23 | -12.49-48.69j
24 | -12.50-98.95j


Numpy fft
0 | 300.00+0.00j
1 | -12.50+98.95j
2 | -12.50+48.68j
3 | -12.50+31.57j
4 | -12.50+22.74j
5 | -12.50+17.20j
6 | -12.50+13.31j
7 | -12.50+10.34j
8 | -12.50+7.93j
9 | -12.50+5.88j
10 | -12.50+4.06j
11 | -12.50+2.38j
12 | -12.50+0.79j
13 | -12.50-0.79j
14 | -12.50-2.38j
15 | -12.50-4.06j
16 | -12.50-5.88j
17 | -12.50-7.93j
18 | -12.50-10.34j
19 | -12.50-13.31j
20 | -12.50-17.20j
21 | -12.50-22.74j
22 | -12.50-31.57j
23 | -12.50-48.68j
24 | -12.50-98.95j


In [20]:
digit_reversed_data = np.array([0, 5, 10, 15, 20, 1, 6, 11, 16, 21, 2, 7, 12, 17, 22, 3, 8, 13, 18, 23, 4, 9, 14, 19, 24])
radix = 5

# Perform reverse digit-reversal
original_data = reverse_digit_reverse_array(digit_reversed_data, radix)
print("Digit-Reversed Data:", digit_reversed_data)
print("Restored Original Data:", original_data)

Digit-Reversed Data: [ 0  5 10 15 20  1  6 11 16 21  2  7 12 17 22  3  8 13 18 23  4  9 14 19
 24]
Restored Original Data: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24]


## Digit inversion

In [49]:
def digit_reverse(radices):
    radices_rev = np.copy(radices)
    radices_rev = radices_rev[::-1]
    radices_len = len(radices)

    mr_fft_len = np.prod(radices)
    indices = np.arange(mr_fft_len)

    mult_factors = []
    tmp_len = mr_fft_len
    for radix in radices:
        tmp_len = tmp_len//radix
        mult_factors.append(tmp_len)
    
    mult_factors_rev = []
    tmp_len = mr_fft_len
    for radix in radices_rev:
        tmp_len = tmp_len//radix
        mult_factors_rev.append(tmp_len)
    

    decomp = []
    for index in indices:
        tmp_decomposition = []
        tmp_index = index
        for mult_factor in mult_factors:
            tmp_decomposition.append(tmp_index // mult_factor)
            tmp_index = tmp_index % mult_factor
        decomp.append(tmp_decomposition)
    
    decomp = np.array(decomp)
    decomp = decomp.T[::-1]
    # print(decomp)


    indices_rev = np.multiply(np.tile(mult_factors_rev, (mr_fft_len,1)), decomp.T).sum(axis=1)

    return indices_rev

In [ ]:
def digit_reverse(radices):
    radices_rev = np.copy(radices)
    radices_rev = radices_rev[::-1]
    radices_len = len(radices)

    mr_fft_len = np.prod(radices)
    indices = np.arange(mr_fft_len)

    mult_factors = []
    tmp_len = mr_fft_len
    for radix in radices:
        tmp_len = tmp_len//radix
        mult_factors.append(tmp_len)
    
    mult_factors_rev = []
    tmp_len = mr_fft_len
    for radix in radices_rev:
        tmp_len = tmp_len//radix
        mult_factors_rev.append(tmp_len)
    

    decomp = []
    for index in indices:
        tmp_decomposition = []
        tmp_index = index
        for mult_factor in mult_factors:
            tmp_decomposition.append(tmp_index // mult_factor)
            tmp_index = tmp_index % mult_factor
        decomp.append(tmp_decomposition)
    
    decomp = np.array(decomp)
    decomp = decomp.T[::-1]
    print(decomp)


    indices_rev = np.multiply(np.tile(mult_factors_rev, (mr_fft_len,1)), decomp.T).sum(axis=1)

    return indices_rev

## Decompositon of N on $2^i \cdot 3^j \cdot 5^k$

The max number of subcarriers in OFDM (5G NR standard) is 3300 $(275 * 12)$, so besides radix 2, 3 and 5, a radix 11 butterfly ($275 = 5^2 \cdot 11^1$) is also needed to achieve the best performance (lowest spectral leakage).
In the next few cells radix powers and the list of all possible FFT sizes will be generated.

It is also possible to avoid the radix 11 butterfly usage if the system is willing to introduce some error because of the spectral leakage. In that case, radices 2, 3 and 5 could generate FFT of size 3840.

In [22]:
i_arr = []
j_arr = []
k_arr = []
for i in range(10):
    for j in range(10):
        for k in range(10):
            if ((((2**i) * (3**j) * (5**k)) <= 275)):
                i_arr.append(i)
                j_arr.append(j)
                k_arr.append(k)

In [23]:
N_arr = []
i_arr = np.unique(i_arr)
j_arr = np.unique(j_arr)
k_arr = np.unique(k_arr)

for i in i_arr:
    for j in j_arr:
        for k in k_arr:
            if((12 * ((2**i) * (3**j) * (5**k))) <= 3300):
            # if (not ((12 * (2**i * 3**j * 5**k)) in N_arr)):
                # print(f"i = {i}, j = {j}, k = {k}")
                N_arr.append(12 * (2**i * 3**j * 5**k))

N_arr.sort()

print(max(N_arr))
print(len(N_arr))

print(N_arr)
print(i_arr)
print(j_arr)
print(k_arr)


3240
53
[np.int64(12), np.int64(24), np.int64(36), np.int64(48), np.int64(60), np.int64(72), np.int64(96), np.int64(108), np.int64(120), np.int64(144), np.int64(180), np.int64(192), np.int64(216), np.int64(240), np.int64(288), np.int64(300), np.int64(324), np.int64(360), np.int64(384), np.int64(432), np.int64(480), np.int64(540), np.int64(576), np.int64(600), np.int64(648), np.int64(720), np.int64(768), np.int64(864), np.int64(900), np.int64(960), np.int64(972), np.int64(1080), np.int64(1152), np.int64(1200), np.int64(1296), np.int64(1440), np.int64(1500), np.int64(1536), np.int64(1620), np.int64(1728), np.int64(1800), np.int64(1920), np.int64(1944), np.int64(2160), np.int64(2304), np.int64(2400), np.int64(2592), np.int64(2700), np.int64(2880), np.int64(2916), np.int64(3000), np.int64(3072), np.int64(3240)]
[0 1 2 3 4 5 6 7 8]
[0 1 2 3 4 5]
[0 1 2 3]


# Combining the stages with different radices

## Radix 3 and radix 2 stage

In [50]:
N = 18
test_vector = np.random.random(N)

fft_stage0 = radix3_SDF_stage(stage_index=0, num_of_stages=1, size=18)
fft_stage1 = radix3_SDF_stage(stage_index=0, num_of_stages=1, size=6)
fft_stage2 = radix2_SDF_stage(stage_index=0, num_of_stages=1)

test_vector_padded = np.append(test_vector, np.zeros(17))
fft_mixed_radix_manual = []
for i in range(len(test_vector_padded)):
    fft_stage0.input_sample = test_vector_padded[i]
    fft_stage0.calculate()
    fft_stage1.input_sample = fft_stage0.output_sample
    fft_stage1.calculate()
    fft_stage2.input_sample = fft_stage1.output_sample
    fft_stage2.calculate()
    print(fft_stage2.output_sample)
    if i >= 17:
        fft_mixed_radix_manual.append(fft_stage2.output_sample)

fft_mixed_radix_manual = np.array(fft_mixed_radix_manual)

radices = [3,3,2]
rev_seq = digit_reverse(radices)

fft_mixed_radix_manual_rev = np.zeros(N).astype(complex)
i = 0
while (i < N):
    fft_mixed_radix_manual_rev[rev_seq[i]] = fft_mixed_radix_manual[i]
    i += 1

k =  0
k =  1
k =  2
k =  3
k =  4
k =  5
k =  0
k =  2
k =  4
k =  6
k =  8
k =  10
STAGE 0, twiddle = [ 1.        +0.j          1.        +0.j          1.        +0.j
  1.        +0.j          1.        +0.j          1.        +0.j
  1.        +0.j          0.93969262-0.34202014j  0.76604444-0.64278761j
  0.5       -0.8660254j   0.17364818-0.98480775j -0.17364818-0.98480775j
  1.        +0.j          0.76604444-0.64278761j  0.17364818-0.98480775j
 -0.5       -0.8660254j  -0.93969262-0.34202014j -0.93969262+0.34202014j]
k =  0
k =  1
k =  0
k =  2
STAGE 0, twiddle = [ 1. +0.j         1. +0.j         1. +0.j         0.5-0.8660254j
  1. +0.j        -0.5-0.8660254j]
0.0
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
(9.064171285973158+0j)
(0.15090161293940874+0j)
(-0.2559946055074486+0.3894097429609816j)
(-0.9721239978455924+0.2124951919936851j)
(-0.9721239978455924-0.21249519199368522j)
(-0.2559946055074486-0.38940974296098146j)
(-0.8208608269797812+0.21878006071181744j)
(1.06253142549

In [51]:
print("Manual fft mixed radix")
i = 0
for num in fft_mixed_radix_manual_rev:
    print(f'{i} | {num:.2f}')
    i+=1

fft_mixed_radix_numpy = np.fft.fft(test_vector)
print("\n")

print("Numpy fft")
i=0
for num in fft_mixed_radix_numpy:
    print(f'{i} | {num:.2f}')
    i+=1

Manual fft mixed radix
0 | 9.06+0.00j
1 | -0.82+0.22j
2 | 0.84+0.93j
3 | -0.26+0.39j
4 | -0.33-0.15j
5 | 0.84+1.52j
6 | -0.97-0.21j
7 | -0.17-1.56j
8 | 1.06-1.10j
9 | 0.15+0.00j
10 | 1.06+1.10j
11 | -0.17+1.56j
12 | -0.97+0.21j
13 | 0.84-1.52j
14 | -0.33+0.15j
15 | -0.26-0.39j
16 | 0.84-0.93j
17 | -0.82-0.22j


Numpy fft
0 | 9.06+0.00j
1 | -0.82+0.22j
2 | 0.84+0.93j
3 | -0.26+0.39j
4 | -0.33-0.15j
5 | 0.84+1.52j
6 | -0.97-0.21j
7 | -0.17-1.56j
8 | 1.06-1.10j
9 | 0.15+0.00j
10 | 1.06+1.10j
11 | -0.17+1.56j
12 | -0.97+0.21j
13 | 0.84-1.52j
14 | -0.33+0.15j
15 | -0.26-0.39j
16 | 0.84-0.93j
17 | -0.82-0.22j


In [26]:
print("Manual fft mixed radix")
i = 0
for num in np.sort(fft_mixed_radix_manual):
    print(f'{i} | {num:.2f}')
    i+=1
# fft_radix3_numpy = np.fft.fft([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0])
print('\n')
print("Numpy fft")
i=0
for num in np.sort(fft_mixed_radix_numpy):
    print(f'{i} | {num:.2f}')
    i+=1

Manual fft mixed radix
0 | -1.22+1.44j
1 | -1.22-1.44j
2 | 0.22+0.89j
3 | 0.22-0.89j
4 | 0.23+0.17j
5 | 0.23-0.17j
6 | 0.50+1.91j
7 | 0.50-1.91j
8 | 0.74-0.51j
9 | 0.74+0.51j
10 | 0.76-0.56j
11 | 0.76+0.56j
12 | 0.79-0.55j
13 | 0.79+0.55j
14 | 1.19-0.68j
15 | 1.19+0.68j
16 | 1.73+0.00j
17 | 7.40+0.00j


Numpy fft
0 | -1.22-1.44j
1 | -1.22+1.44j
2 | 0.22+0.89j
3 | 0.22-0.89j
4 | 0.23-0.17j
5 | 0.23+0.17j
6 | 0.50-1.91j
7 | 0.50+1.91j
8 | 0.74-0.51j
9 | 0.74+0.51j
10 | 0.76-0.56j
11 | 0.76+0.56j
12 | 0.79+0.55j
13 | 0.79-0.55j
14 | 1.19-0.68j
15 | 1.19+0.68j
16 | 1.73-0.00j
17 | 7.40+0.00j


## Radix 5 and radix 2

In [46]:
N = 10
test_vector = np.random.random(N)

radices = [5,2]
rev_seq = digit_reverse(radices)

fft_stage0 = radix5_SDF_stage(stage_index=0, num_of_stages=1, size=10)
fft_stage1 = radix2_SDF_stage(stage_index=0, num_of_stages=1)

test_vector_padded = np.append(test_vector, np.zeros(9))
fft_mixed_radix_manual = []
for i in range(len(test_vector_padded)):
    fft_stage0.input_sample = test_vector_padded[i]
    fft_stage0.calculate()
    fft_stage1.input_sample = fft_stage0.output_sample
    fft_stage1.calculate()
    print(fft_stage1.output_sample)
    if i >= 9:
        fft_mixed_radix_manual.append(fft_stage1.output_sample)

fft_mixed_radix_manual = np.array(fft_mixed_radix_manual)

fft_mixed_radix_manual_rev = np.zeros(N).astype(complex)
i = 0
while (i < N):
    fft_mixed_radix_manual_rev[rev_seq[i]] = fft_mixed_radix_manual[i]
    i += 1

[[0 1 0 1 0 1 0 1 0 1]
 [0 0 1 1 2 2 3 3 4 4]]
k =  0
k =  1
k =  0
k =  2
k =  0
k =  3
k =  0
k =  4
STAGE 0, twiddle = [ 1.        +0.j          1.        +0.j          1.        +0.j
  0.80901699-0.58778525j  1.        +0.j          0.30901699-0.95105652j
  1.        +0.j         -0.30901699-0.95105652j  1.        +0.j
 -0.80901699-0.58778525j]
0.0
0j
0j
0j
0j
0j
0j
0j
0j
(4.998975266773563+0j)
(0.10723308405830068+0j)
(-0.6376922735671277-0.4886544088472911j)
(-0.84505070760013+0.599126998474012j)
(0.401943459869859-0.7988448632033959j)
(-0.5089199918838262+0.2099974412053801j)
(-0.5089199918838261-0.20999744120538022j)
(0.40194345986985897+0.798844863203396j)
(-0.8450507076001299-0.599126998474012j)
(-0.6376922735671278+0.4886544088472911j)


In [45]:
print("Manual fft mixed radix")
# fft_mixed_radix_manual = fft_mixed_radix_manual[rev_seq]
i = 0
for num in fft_mixed_radix_manual_rev:
    print(f'{i} | {num:.2f}')
    i+=1

fft_mixed_radix_numpy = np.fft.fft(test_vector)
print("\n")

print("Numpy fft")
i=0
for num in fft_mixed_radix_numpy:
    print(f'{i} | {num:.2f}')
    i+=1

Manual fft mixed radix
0 | 5.18+0.00j
1 | 0.20-1.22j
2 | -0.18-0.26j
3 | -0.84-0.52j
4 | 1.00-1.36j
5 | -1.02+0.00j
6 | 1.00+1.36j
7 | -0.84+0.52j
8 | -0.18+0.26j
9 | 0.20+1.22j


Numpy fft
0 | 5.18+0.00j
1 | 0.20-1.22j
2 | -0.18-0.26j
3 | -0.84-0.52j
4 | 1.00-1.36j
5 | -1.02-0.00j
6 | 1.00+1.36j
7 | -0.84+0.52j
8 | -0.18+0.26j
9 | 0.20+1.22j
